We are implementing the **Avellaneda-Stoikov (AS)** (pronunciation: Ah-veh-yah-neh-DAH STOY-koff) Model, which is the gold standard for high-frequency market making. It answers two critical questions:
- Where should I center my quotes? (Reservation Price)
- How far apart should my Bid and Ask be? (Optimal Spread)

Ref: https://people.orie.cornell.edu/sfs33/LimitOrderBook.pdf

#### 1. The Reservation Price ($r$)

In a perfect world, you’d just quote around the Mid-price. But if you already own 100 bonds (Inventory $q = 100$), you are "at risk." If the price drops, you lose money.

The **Reservation Price** is your "Personal Mid-Price." It skews your quotes to help you get back to zero inventory.

$$r = s - q \gamma \sigma^2 (T - t)$$

* **$s$ (Mid/Micro Price):** The current market price.
* **$q$ (Inventory):** Your current position (e.g., +100 shares).
* **$\gamma$ (Risk Aversion):** Your "fear" level. High $\gamma$ makes you move $r$ faster to dump inventory.
* **$\sigma^2$ (Variance):** Market volatility. Risk grows as the market gets crazier.
* **$(T-t)$ (Time Horizon):** Time remaining. Usually kept at **1.0** for a continuous "rolling" risk window.

> **Logic:** If you are **Long** ($q > 0$), $r$ becomes **lower** than the market price. This makes your **Ask** cheaper (so people buy from you) and your **Bid** lower (so people don't sell to you). You are essentially "leaning" the book to dump your risk.

---

#### 2. The Optimal Spread ($\delta$)

This determines the width between your Bid and Ask. It’s a balance:

* **Too narrow:** You get filled often, but you lose money to "Toxic Flow" (Kyle's Lambda) and Volatility.
* **Too wide:** You are safe, but no one ever trades with you.

$$\delta = \gamma \sigma^2 (T-t) + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa}\right)$$

* **Volatility Component ($\gamma \sigma^2$):** The spread widens linearly with volatility. This is your "safety buffer" against price jumps.
* **Liquidity Component ($\kappa$):** Represents how quickly your "fill probability" drops as you move away from the Mid.
* **$\kappa$**: Market Liquidity (how likely an order is to hit your quote).
* * **High $\kappa$ (Liquid):** Many traders; you can quote tight.
* * **Low $\kappa$ (Illiquid):** Few traders; you must quote wider because getting "hit" is a rarer, riskier event.

---

#### 3. The Final Quotes

Once the engine calculates $r$ and $\delta$, it outputs the two prices you will actually send to the exchange:

* **Your Bid** $= r - \frac{\delta}{2}$
* **Your Ask** $= r + \frac{\delta}{2}$

##### Q1: Is $\gamma$ (Risk Aversion) a constant, or do I change it?
- In professional systems, $\gamma$ is a "risk appetite" setting.
- **Scenario:** Your firm is having a bad month and can't afford more losses.
- **Action:** You crank up $\gamma$. The bot becomes extremely "shy." It widens spreads and refuses to hold inventory for more than a few seconds. It makes less profit, but the Drawdown is smaller.

##### Q2: How do I actually find a value for $\kappa$ (Liquidity)?
- This is the hardest part of the AS model. $\kappa$ essentially models the intensity of trades.
- Technically, it represents the probability of an order hitting your quote: $P(\text{fill}) \propto e^{-\kappa \delta}$.

In [2]:
# Let kappa and risk aversion be our "knobs"

In [3]:
# Implementation of reservation price
def reservation_price(s, q, gamma, variance, time_horizon):
    return s - q*gamma*variance*time_horizon

reservation_price(99.8, 100, 0.1, 0.02, 1)

99.6

In [7]:
import numpy as np

def optimal_spread(gamma, variance, time_horizon, kappa):
    # risk_term (charges for volatility) + liquidity_term ("minimal" addition to attract fill)
    return gamma*variance*time_horizon + (2*np.log(1+gamma/kappa))/gamma


optimal_spread(0.1, 0.02, 1, 2)

0.977803283388641

In [10]:
def optimal_spread(gamma, variance, time_horizon, kappa):
    # risk_term (charges for volatility) + liquidity_term ("minimal" addition to attract fill)
    risk_term = gamma*variance*time_horizon
    liquidity_term = (2*np.log(1+gamma/kappa))/gamma
    return risk_term+liquidity_term, risk_term, liquidity_term


optimal_spread(0.1, 0.02, 1, 2)

(0.977803283388641, 0.002, 0.975803283388641)

In [17]:
prices = [100.00, 100.02, 100.01, 100.03, 100.00, # Stable
          100.05, 100.10, 100.12, 100.08, 100.15, # Trending
          100.40, 100.80, 101.50, 101.30, 101.90, # Breakout
          101.85, 101.90, 101.88, 101.92, 101.90] # Plateau

np.diff(prices)

array([ 0.02, -0.01,  0.02, -0.03,  0.05,  0.05,  0.02, -0.04,  0.07,
        0.25,  0.4 ,  0.7 , -0.2 ,  0.6 , -0.05,  0.05, -0.02,  0.04,
       -0.02])

In [18]:
len(np.diff(prices))

19

In [22]:
f"{np.var(np.diff(np.log(prices))):.8f}"

'0.00000485'

In [16]:

gamma = 50
kappa = 0.1
window = 5

for i in range(window, len(prices)):
    # Calculate rolling variance of the window
    window_prices = prices[i-window:i]
    returns = np.diff(np.log(window_prices))
    variance = np.var(returns)
    
    # Calculate AS Spread
    opt_spread, risk_term, liquidity_term = optimal_spread(gamma, variance, 1, kappa)
    
    print(f"Step:{i:<5} | Price:{prices[i]:<8.2f} | Rolling Vol: {variance:<12.8f} | Optimal Spread: {opt_spread:<12.4f} ({risk_term:8f} {liquidity_term:.8f})")

Step:5     | Price:100.05   | Rolling Vol: 0.00000004   | Optimal Spread: 0.2487       (0.000002 0.24866424)
Step:6     | Price:100.10   | Rolling Vol: 0.00000009   | Optimal Spread: 0.2487       (0.000005 0.24866424)
Step:7     | Price:100.12   | Rolling Vol: 0.00000011   | Optimal Spread: 0.2487       (0.000005 0.24866424)
Step:8     | Price:100.08   | Rolling Vol: 0.00000011   | Optimal Spread: 0.2487       (0.000005 0.24866424)
Step:9     | Price:100.15   | Rolling Vol: 0.00000013   | Optimal Spread: 0.2487       (0.000007 0.24866424)
Step:10    | Price:100.40   | Rolling Vol: 0.00000017   | Optimal Spread: 0.2487       (0.000009 0.24866424)
Step:11    | Price:100.80   | Rolling Vol: 0.00000117   | Optimal Spread: 0.2487       (0.000058 0.24866424)
Step:12    | Price:101.50   | Rolling Vol: 0.00000280   | Optimal Spread: 0.2488       (0.000140 0.24866424)
Step:13    | Price:101.30   | Rolling Vol: 0.00000520   | Optimal Spread: 0.2489       (0.000260 0.24866424)
Step:14    | Price:

- Quiet Days: Spread is determined entirely by $\kappa$ (how much you need to charge just to exist in the market).
- Panic Days: The volatility term suddenly explodes and becomes the dominant force, forcing the bot to double or triple its spread.

#### Impact of "Inventory Skew" ($q$)

In [29]:
def get_quotes(s, q, gamma, variance, time_horizon, kappa):
    # r
    res_price = reservation_price(s, q, gamma, variance, time_horizon) 
    
    # delta
    spread, _, _ = optimal_spread(gamma, variance, time_horizon, kappa)

    # quotes
    bid = res_price - (spread / 2)
    ask = res_price + (spread / 2)
    
    return bid, ask

In [37]:
mid_price = 100.00
gamma = 0.5
variance = 0.001
kappa = 0.2

inventory_levels = [-100, -50, 0, 50, 100]

for q in inventory_levels:
    b, a = get_quotes(mid_price, q, gamma, variance, 1, kappa)
    print(f"Inventory {q:<15} | r={(a+b)/2:<10.2f} | bid={b:<10.2f} | ask={a:<10.2f}")

Inventory -100            | r=100.05     | bid=97.54      | ask=102.56    
Inventory -50             | r=100.03     | bid=97.52      | ask=102.53    
Inventory 0               | r=100.00     | bid=97.49      | ask=102.51    
Inventory 50              | r=99.97      | bid=97.47      | ask=102.48    
Inventory 100             | r=99.95      | bid=97.44      | ask=102.46    


In [38]:
# Negative invetory -100 means we are short of shares -> intuition is to attract sellers -> offer good price -> price rise!
# Positive inventory +100 -> abundant shares -> attract buyers -> offer relaxed price -> price fall!

Summary: 
- When you are Long (+100): Your Reservation Price $r$ drops. Your Ask becomes very attractive (close to 100 or below) because you want to sell. Your Bid drops way down because you don't want to buy any more!
  
- When you are Short (-100): The opposite happens. $r$ jumps up. You raise your Bid to attract sellers so you can "cover" your short, and you raise your Ask so high that no one will buy from you.